# 历史成员身份与有限补数
只对同一发行人的更名前历史尝试真实查询，不用收购方价格替代退市股票，不拼接WOLF新旧证券。所得候选行情独立保留，既有来源不改。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.market_update import _fetch_yahoo_chart_payload
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_identity_repair_20261014';plan=json.loads((card/'plan.json').read_text());rawdir=root/plan['raw_directory'];rawdir.mkdir(parents=True,exist_ok=True);payload_path=rawdir/'AZTA_chart.json';receipt_path=rawdir/'receipt.json'
if not payload_path.exists():
 payload=_fetch_yahoo_chart_payload('AZTA',start=plan['start'],end=plan['end'])
 atomic_json(payload,payload_path);atomic_json({'retrieved_at':datetime.now(timezone.utc).isoformat(),'query_symbol':'AZTA','requested_start':plan['start'],'requested_end':plan['end'],'source_api':'Yahoo chart v8 through existing market_update helper','payload_sha256':hashlib.sha256(payload_path.read_bytes()).hexdigest(),'original_historical_ticker_metadata_available':False},receipt_path)
payload=json.loads(payload_path.read_text());receipt=json.loads(receipt_path.read_text());assert hashlib.sha256(payload_path.read_bytes()).hexdigest()==receipt['payload_sha256']
result=payload['chart']['result'][0];quotes=result['indicators']['quote'][0];adj=result['indicators']['adjclose'][0]['adjclose'];dates=pd.to_datetime(result['timestamp'],unit='s',utc=True).tz_convert('America/New_York').normalize().tz_localize(None)
frame=pd.DataFrame({'datetime':dates,'instrument':'BRKS',**{name:quotes[name] for name in ['open','high','low','close','volume']},'adj_close':adj}).sort_values('datetime')
assert not frame.datetime.duplicated().any() and frame.datetime.between(plan['start'],plan['end']).all()
assert np.isfinite(frame[['open','high','low','close','volume','adj_close']]).all().all()
assert frame[['open','high','low','close','adj_close']].gt(0).all().all() and frame.volume.ge(0).all()
assert frame.high.ge(frame[['open','close','low']].max(axis=1)).all() and frame.low.le(frame[['open','close','high']].min(axis=1)).all()
import exchange_calendars as xc
calendar=xc.get_calendar('XNYS',start=plan['start'],end=plan['end']).schedule.index
assert pd.DatetimeIndex(frame.datetime).equals(calendar)
provider=root/plan['provider'];manifest=build_qlib_store_from_frame(frame,provider_uri=provider,freq='day',as_of=plan['end'],field_map={x:x for x in ['open','high','low','close','volume','adj_close']},source='Yahoo AZTA issuer-continuity candidate history for historical BRKS; SEC rename reference in fixed plan; not independently price-certified')
qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['BRKS'],fields=['$close','$adj_close'],start=plan['start'],end=plan['end']).xs('BRKS',level='instrument')
np.testing.assert_allclose(native['$close'],frame.close,rtol=2e-7,atol=1e-5)
np.testing.assert_allclose(native['$adj_close'],frame.adj_close,rtol=2e-7,atol=1e-5)
parent=root/'research/constituent_breadth_source_20261014';ps=json.loads((parent/'summary.json').read_text());pm=parent/'verification_manifest.json';assert hashlib.sha256(pm.read_bytes()).hexdigest()==plan['parent_manifest_sha256']
with R.uri_context(ps['recorder_uri']):
 pr=R.get_recorder(recorder_id=plan['parent_recorder_id'],experiment_name=parent.name);entry=next(e for e in json.loads(pm.read_text())['entries'] if e['path'].endswith('/artifacts/analysis/missing'));path=Path(pr.download_artifact('analysis/missing'));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];missing=pr.load_object('analysis/missing')
needed=missing.loc[missing.symbol.eq('BRKS')].copy();prices=native['$adj_close'];window_ready=prices.rolling(20,min_periods=20).count().eq(20)&prices.shift(5).notna();needed['candidate_window_ready']=needed.date.map(window_ready).fillna(False)
assert needed.candidate_window_ready.all()
summary={'experiment':card.name,'decision':'BRKS_alias_history_candidate_recovered_not_merged','new_fits':0,'new_backtests':0,'outcome_correlations':0,'objective_achieved':False,'query_symbol':'AZTA','historical_instrument':'BRKS','historical_cusip':'114340102','identity_reference':plan['identity_reference'],'identity_role':'Rename continuity supported; Yahoo payload itself does not expose per-date CUSIP','rows':len(frame),'first_price_date':str(frame.datetime.min().date()),'last_price_date':str(frame.datetime.max().date()),'previous_missing_BRKS_sessions':len(needed),'candidate_covered_BRKS_sessions':int(needed.candidate_window_ready.sum()),'historical_prices_independently_verified':False,'historical_receipt_certified':False,'canonical_sources_modified':False,'provider_uri':str(provider),'region':'us','as_of':plan['end'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/prices','analysis/covered_sessions']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='identity_bounded_price_intake',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,query_symbol='AZTA',historical_instrument='BRKS')
 rec.save_objects(artifact_path='inputs',plan=plan,receipt=receipt,raw_payload=payload,provider_manifest=manifest)
 rec.save_objects(artifact_path='analysis',prices=native,covered_sessions=needed,summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),candidate_covered_sessions=len(needed))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json')
needed.groupby(needed.date.dt.year).agg(sessions=('candidate_window_ready','size'),covered=('candidate_window_ready','sum'),missing_weight_max=('missing_weight','max')).to_csv(card/'coverage.csv')
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print({k:summary[k] for k in ['rows','first_price_date','last_price_date','candidate_covered_BRKS_sessions','recorder_id']})


In [1]:
from pathlib import Path
import os,json,hashlib,shutil
from datetime import datetime,timezone
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_identity_repair_20261014';plan=json.loads((card/'plan.json').read_text());s=json.loads((card/'summary.json').read_text());rawdir=root/plan['raw_directory'];qlib.init(provider_uri=s['provider_uri'],region='us')
from src.data.qlib_native import QlibProviderConfig,load_qlib_feature_frame
native=load_qlib_feature_frame(QlibProviderConfig(Path(s['provider_uri'])),instruments=['BRKS'],fields=['$close'],start=plan['start'],end=plan['end']).xs('BRKS',level='instrument')
source_manifest=json.loads((root/'data/qlib/soxx_pit_leveraged_training_v1/sec_nport_holdings_manifest.json').read_text());rows=[];raw_xml={};receipts=[]
for date in ['2020-12-31','2021-03-31']:
 path=rawdir/(date+'.xml');content=path.read_bytes();tree=ET.fromstring(content)
 matches=[node for node in tree.iter() if node.tag.split('}')[-1]=='invstOrSec' and any(e.tag.split('}')[-1]=='cusip' and e.text=='114340102' for e in node.iter())]
 assert len(matches)==1;values={e.tag.split('}')[-1]:e.text for e in matches[0]};assert values['units']=='NS' and values['curCd']=='USD' and values['assetCat']=='EC' and values['fairValLevel']=='1'
 implied=float(values['valUSD'])/float(values['balance']);quote=float(native.loc[date,'$close']);difference=abs(implied-quote)
 assert difference<1e-4
 filing=next(e for e in source_manifest['filings'] if e['report_date']==date)
 rows.append({'report_date':date,'cusip':values['cusip'],'reported_shares':float(values['balance']),'reported_value_usd':float(values['valUSD']),'implied_price_usd':implied,'native_candidate_close':quote,'absolute_difference':difference,'source_url':filing['primary_xml_url']})
 raw_xml[date]=content;receipts.append({'report_date':date,'source_url':filing['primary_xml_url'],'http_status':200,'sha256':hashlib.sha256(content).hexdigest(),'raw_saved_at':datetime.fromtimestamp(path.stat().st_mtime,timezone.utc).isoformat()})
s['two_reported_close_anchors_verified']=True;s['anchor_count']=2;s['maximum_anchor_difference']=max(r['absolute_difference'] for r in rows);s['decision']='BRKS_alias_history_two_anchor_validated_not_merged';s['anchor_scope']='Independent reported fair-value/share checks for two dates only; not full-series or historical-vintage certification'
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=card.name)
 rec.save_objects(artifact_path='inputs',sec_anchor_xml=raw_xml,sec_anchor_receipts=receipts)
 rec.save_objects(artifact_path='analysis',anchor_checks=pd.DataFrame(rows),summary=s)
 artifacts=Path(s['recorder_uri'])/s['experiment_id']/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),verified_close_anchors=2)
(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
s['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=s['experiment_id']);(card/'summary.json').write_text(json.dumps(s,indent=2)+'\n');pd.DataFrame(rows).to_csv(card/'price_anchor_checks.csv',index=False);print(pd.DataFrame(rows).to_string(index=False))


# 历史成员身份：BRKS补数取得可核查进展

本轮0拟合、0回测、0未来收益相关筛选。官方文件区分了更名与收购/注销，避免用买方价格填补被收购证券，或把相同ticker的新旧证券视为连续一股。BRKS→AZTA的公司更名2021-12-01生效，SEC文件明确股东权利不受更名影响；据此只查询更名前的BRKS持仓窗口。

既有Yahoo接口通过AZTA符号取得2020-10-01至2021-08-27共229条日线，实际NYSE日历无缺失，OHLC/正价/成交量检查通过。独立Qlib provider以历史BRKS标识读回，源manifest注明查询别名AZTA及历史CUSIP114340102；并非生成或拼接价格。

候选历史覆盖此前BRKS缺失的188个成员日（2020年24、2021年164），均有20日回看窗口。其他缺失成员仍存在，不能称整个成分篮子已完整。旧provider、NAS当前核心来源、冻结模型和生产release均未改。

进一步取得2020-12-31、2021-03-31两份原始SEC N-PORT XML。对CUSIP114340102，核实单位NS、USD、一级公允价值，用市值/股数独立得到每股67.85和81.65美元；与候选Qlib收盘价最大差0.00000153美元，仅浮点精度量级。原XML、请求来源和字节哈希已入NAS Recorder。

两个日期锚点支持这段行情确属原BRKS，但不认证每个交易日、历史首次接收或当时复权版本；Yahoo载荷没有逐日CUSIP。决定：BRKS_alias_history_two_anchor_validated_not_merged。保留为可追溯补数来源，不覆盖原冻结历史。

身份核查的其他边界：CREE在2021-10-04更名为WOLF；2025-09-29旧WOLF股份被注销/转换，新证券仍使用WOLF代码但CUSIP不同，不能直接补到旧成员历史。XLNX→AMD及MXIM→ADI为收购换股；IPHI交易含现金和MRVL股份；CCMP曾更名并后被收购。它们均需原证券历史或明确公司行动账本，不可用收购方当前曲线反填。

下一步将已核实的BRKS来源用于新的披露篮子覆盖核对，继续针对旧CUSIP977852102核查WOLF历史；保留缺失边界，不用上市新股价格冒充旧股，不因无法补齐而删除压力区间。稳定净超额仍未达到，本轮仅取得真实数据与身份核对进展。

[成员日覆盖](coverage.csv) · [SEC价格锚点](price_anchor_checks.csv) · [证券身份参考](identity_review.csv)

Recorder `0a52125bcfb04f9b9217e287227ab607`。


## 来源范围更正（2026-10-14）
本卡的缺价统计只针对主provider，不能代表整个工作区缺少旧WOLF历史。后续核查找到10月5日已恢复并经SEC锚点核对的独立WOLF/CREE快照，以及当时已关闭的semiconductor_breadth_20261005实验。补充读取后2023Q4为63/63完整披露篮子日。原统计不改，补充来源与纠正证据见[来源衔接卡](../constituent_source_reconciliation_20261014/conclusion.md)。不得再把旧WOLF获取或旧四特征广度路线称作尚未做过。
